# TTFT와 total 비교

같은 에이전트를 세 모드로 실행할 때 첫 사건, 첫 토큰, 전체 완료 시간은 어떻게 달라질까요?


## 실행 환경 준비

Python 3.12와 고정된 패키지 환경을 사용합니다. 키 값은 출력하지 않고 존재 여부만 확인합니다.


In [1]:
# 고정한 패키지를 설치하고 비밀 키의 존재 여부만 확인합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3
from pathlib import Path
import os
from importlib.metadata import version
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break

MODEL = "gpt-5.4-mini"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
print({"langchain": version("langchain"), "langgraph": version("langgraph"), "model": MODEL, "cost_limit_usd": 1.0})


Note: you may need to restart the kernel to use updated packages.
{'langchain': '1.4.0', 'langgraph': '1.2.11', 'model': 'gpt-5.4-mini', 'cost_limit_usd': 1.0}


## 측정 그래프 준비

### 코드 읽기와 비교 경계

`perf_counter()`는 시스템 시계가 수정되어도 경과 시간 측정에 사용할 수 있는 단조 증가 시계를 읽습니다. `measure()`는 스트림을 만들기 직전에 시작 시각을 기록하고, 첫 `yield`에서 `first_event`, `messages`의 내용 있는 첫 조각에서 `ttft_s`, 반복문이 끝난 뒤 `total_s`를 계산합니다. 함수가 반환하는 딕셔너리의 `None`은 측정 실패가 아니라 해당 모드에는 답변 조각이라는 관찰 대상이 없었다는 표시입니다.

세 모드의 시간을 공정하게 비교하려면 한 모델 호출에서 여러 모드를 동시에 구독하거나, 실행 순서를 섞어 충분히 반복해야 합니다. 현재 코드는 모드마다 새 모델 요청을 보내므로 네트워크 상태와 생성된 답변 길이도 함께 달라집니다. `custom`의 첫 사건이 매우 빨리 도착해도 그것은 ‘작업 시작’을 일찍 알린 결과이며 모델의 첫 답변이 빨라졌다는 뜻이 아닙니다. 반대로 `messages`의 `total`이 짧게 나온 한 번의 결과도 스트리밍이 전체 계산 시간을 줄였다는 증거가 되지 않습니다.


In [2]:
# 같은 모델과 질문을 세 모드에서 측정할 그래프를 구성합니다.
from time import perf_counter
from typing import TypedDict
from langchain.chat_models import init_chat_model
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    answer: str

call_records = []
model = init_chat_model(f"openai:{MODEL}", reasoning_effort="none")
def answer(state: State) -> dict[str, str]:
    get_stream_writer()({"stage": "started"})
    response = model.invoke("세 문장 이내로 답하세요: " + state["question"])
    assert response.usage_metadata is not None
    call_records.append({"answer": response.text, "usage": response.usage_metadata})
    return {"answer": response.text}

builder = StateGraph(State)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()


## 세 번의 독립 실행 준비

같은 질문을 updates, messages, custom 순서로 한 번씩 실행합니다. 호출마다 답변 길이와 서버 상태가 달라질 수 있으므로 모드가 속도 차이의 원인이라고 판단하지 않습니다.

각 함수 호출은 독립된 그래프 실행입니다. 따라서 세 수치의 차이에는 모드 외에도 네트워크와 모델 상태가 함께 들어갑니다.


In [3]:
# 한 모드를 끝까지 소비하고 클라이언트에서 관측한 시간을 반환합니다.
def measure(mode: str, question: str) -> dict:
    started = perf_counter()
    first_event = None
    first_text = None
    event_count = 0
    for data in graph.stream({"question": question, "answer": ""}, stream_mode=mode):
        elapsed = perf_counter() - started
        event_count += 1
        if first_event is None:
            first_event = elapsed
        if mode == "messages":
            chunk, _ = data
            if chunk.text and first_text is None:
                first_text = elapsed
    total = perf_counter() - started
    assert first_event is not None
    return {
        "mode": mode, "first_event_s": round(first_event, 3),
        "ttft_s": None if first_text is None else round(first_text, 3),
        "total_s": round(total, 3), "events": event_count,
    }


## 실제 사용량과 답변 보관

공급자가 반환한 입력·출력 토큰과 캐시 입력 토큰을 사용합니다. 비용은 일반 입력 100만 개당 0.75달러, 캐시 입력 0.075달러, 출력 4.50달러 기준의 추정치입니다. 최종 청구액은 계정의 사용 내역에서 확인합니다.


In [4]:
# 호출별 사용량과 응답 길이를 시간 기록에 함께 붙입니다.
question = "고객에게 LLM 챗봇의 답변 글자를 생성되는 대로 보여 주는 스트리밍의 장점을 설명해 주세요."
rows = []
for mode in ["updates", "messages", "custom"]:
    row = measure(mode, question)
    record = call_records[-1]
    usage = record["usage"]
    cached = usage.get("input_token_details", {}).get("cache_read", 0)
    cost = ((usage["input_tokens"] - cached) * 0.75
            + cached * 0.075 + usage["output_tokens"] * 4.50) / 1_000_000
    row.update({"input_tokens": usage["input_tokens"], "cached_tokens": cached,
                "output_tokens": usage["output_tokens"], "answer_chars": len(record["answer"]),
                "estimated_usd": round(cost, 8)})
    rows.append(row)
    print(row)
    print("실제 답변:", record["answer"])
assert len(call_records) == 3
assert next(r for r in rows if r["mode"] == "messages")["ttft_s"] is not None


{'mode': 'updates', 'first_event_s': 2.622, 'ttft_s': None, 'total_s': 2.622, 'events': 1, 'input_tokens': 45, 'cached_tokens': 0, 'output_tokens': 91, 'answer_chars': 143, 'estimated_usd': 0.00044325}
실제 답변: 스트리밍은 답변이 완성되기 전부터 글자가 차례로 보여져 고객이 더 빠르게 응답을 체감하게 해 줍니다.  
또한 긴 답변을 기다리는 동안의 지루함과 불안감을 줄여 사용 경험을 개선합니다.  
결과적으로 챗봇이 더 빠르고 대화형으로 느껴져 만족도가 높아집니다.


{'mode': 'messages', 'first_event_s': 0.613, 'ttft_s': 0.62, 'total_s': 1.101, 'events': 83, 'input_tokens': 45, 'cached_tokens': 0, 'output_tokens': 84, 'answer_chars': 150, 'estimated_usd': 0.00041175}
실제 답변: 스트리밍은 고객이 답변을 기다리는 시간을 크게 줄여, 챗봇이 더 빠르고 자연스럽게 느껴지게 합니다.  
또한 긴 답변도 생성되는 즉시 확인할 수 있어 사용자가 중간에 방향을 바꾸거나 추가 질문을 하기 쉽습니다.  
결과적으로 응답 속도와 사용자 경험이 함께 개선됩니다.


{'mode': 'custom', 'first_event_s': 0.003, 'ttft_s': None, 'total_s': 1.431, 'events': 1, 'input_tokens': 45, 'cached_tokens': 0, 'output_tokens': 111, 'answer_chars': 179, 'estimated_usd': 0.00053325}
실제 답변: LLM 챗봇의 스트리밍은 답변이 완성되기 전에 글자가 생성되는 대로 바로 보여 주어, 고객이 더 빠르게 응답을 체감할 수 있게 합니다.  
또한 긴 답변도 중간에 끊김 없이 볼 수 있어 대기 시간을 줄이고, 챗봇이 더 즉각적이고 자연스럽게 반응하는 인상을 줍니다.  
결과적으로 사용자 만족도와 대화 몰입도가 높아집니다.


## 지표를 구분해 해석

messages에서 내용이 있는 첫 조각까지의 시간을 TTFT 근삿값으로 사용합니다. 조각 하나와 모델 토큰 하나는 일대일 관계가 아니며, 이 값은 서버 내부의 토큰 생성 시각이 아닙니다.

TTFT는 사용자가 첫 반응을 보는 시점을, total은 모든 사건을 소비한 시점을 근사합니다. 두 지표는 서로 다른 사용자 경험을 답합니다.


In [5]:
# 시작 알림과 답변의 첫 글자를 구분하고 세 호출의 비용을 합칩니다.
for row in rows:
    meaning = "첫 답변 조각의 도착 시간" if row["ttft_s"] is not None else "첫 사건만 측정"
    print(row["mode"], meaning, "total=", row["total_s"])
print("세 호출의 추정 비용(USD):", round(sum(r["estimated_usd"] for r in rows), 8))
print("각 모드를 한 번씩 순서대로 실행했으므로 속도 우열을 판정하지 않습니다.")


updates 첫 사건만 측정 total= 2.622
messages 첫 답변 조각의 도착 시간 total= 1.101
custom 첫 사건만 측정 total= 1.431
세 호출의 추정 비용(USD): 0.00138825
각 모드를 한 번씩 순서대로 실행했으므로 속도 우열을 판정하지 않습니다.
